In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Liberation Serif"
plt.rcParams["font.size"] = 15
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 15
plt.rcParams["xtick.labelsize"] = 15
plt.rcParams["ytick.labelsize"] = 15
plt.rcParams["legend.fontsize"] = 15

In [ ]:
df = pd.read_csv("data_banknote_authentication.txt")

In [ ]:
df.columns.tolist()

In [ ]:
column_names = [
    "Variance",
    "Skewness",
    "Curtosis",
    "Entropy",
    "Class"
]

df = pd.read_csv(
    "data_banknote_authentication.txt",
    header=None,
    names=column_names
)

df.head()

In [ ]:
print("Dataset Shape:", df.shape)

In [ ]:
print(df.isnull().sum())

In [ ]:
df.describe()

In [ ]:
df.hist(figsize=(10,8), bins=20)

plt.suptitle("Feature Histograms")
plt.tight_layout()
plt.savefig(
    "feature_histograms.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

Shows the distribution of each feature in the dataset. Since the data is not perfectly normally distributed some features are skewed. No severe outliers are observed.


In [ ]:
plt.figure(figsize=(7,5))

sns.heatmap(
    df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.savefig(
    "correlation_heatmap.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

Shows the relationships among the input features. Several features show moderate to strong correlations which means they can distinguish between forged and authentic banknotes.

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=df,
    x="Variance",
    y="Skewness",
    hue="Class",
    palette="Set1"
)

plt.title("Variance vs Skewness")
plt.savefig(
    "scatter_plot.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

The two classes exhibit noticeable separation based on the selected features. The data is approximately linearly separable which makes it suitable for a single layer perceptron

In [ ]:
plt.figure(figsize=(12,6))

for i, column in enumerate(df.columns[:-1]):
    plt.subplot(2,2,i+1)
    sns.boxplot(y=df[column])
    plt.title(column)

plt.tight_layout()
plt.savefig(
    "boxplots.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

Shows the spread and variability of each feature and highlights potential outliers. The differences in median values and interquartile ranges show that the features have distinct distributions

In [ ]:
# Features
X = df.iloc[:, :-1]

# Target
y = df.iloc[:, -1]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])


In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
print("Mean of Training Features:")
print(X_train.mean(axis=0))

print("\nStandard Deviation of Training Features:")
print(X_train.std(axis=0))

In [ ]:
import numpy as np

class Perceptron:

    def __init__(self, learning_rate=0.01, epochs=20):
        self.learning_rate = learning_rate
        self.epochs = epochs

    def initialize_parameters(self, n_features):
        self.weights = np.zeros(n_features, dtype=float)
        self.bias = 0.0

    def activation(self, z):
        return 1 if z >= 0 else 0

    def predict_sample(self, x):
        z = np.dot(x, self.weights) + self.bias
        return self.activation(z)

    def fit(self, X, y):
        # Convert pandas objects to NumPy arrays so y[i] uses positions,
        # not the original DataFrame row labels left by train_test_split.
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)

        n_samples, n_features = X.shape
        self.initialize_parameters(n_features)

        self.errors = []
        self.weight_history = []
        self.bias_history = []

        for epoch in range(self.epochs):
            errors = 0

            for i in range(n_samples):
                prediction = self.predict_sample(X[i])
                update = self.learning_rate * (y[i] - prediction)

                self.weights += update * X[i]
                self.bias += update

                if update != 0:
                    errors += 1

            self.errors.append(errors)
            self.weight_history.append(self.weights.copy())
            self.bias_history.append(self.bias)

            print(f"Epoch {epoch + 1:02d} | Misclassified: {errors:3d} | "
                  f"Bias: {self.bias:.4f}")

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self.predict_sample(sample) for sample in X])


In [ ]:


# Create the Perceptron model
perceptron = Perceptron(learning_rate=0.01, epochs=20)

# Train the model
perceptron.fit(X_train, y_train)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(range(1, len(perceptron.errors)+1), perceptron.errors, marker='o')

plt.title("Training Error vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Misclassified Samples")
plt.grid(True)

plt.savefig(
    "training_error_vs_epoch.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

The training error decreases steadily as the number of epochs increases which means that the perceptron is learning from the training data. The convergence of the error shows that the model successfully minimizes misclassifications

In [ ]:
plt.figure(figsize=(8,6))

weight_history = np.array(perceptron.weight_history)

for i in range(weight_history.shape[1]):
    plt.plot(
        range(1, perceptron.epochs + 1),
        weight_history[:, i],
        marker='o',
        label=f'Weight {i+1}'
    )

plt.title("Weight Evolution")
plt.xlabel("Epoch")
plt.ylabel("Weight Value")
plt.legend()
plt.grid(True)

plt.savefig(
    "weight_evolution.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

Shows how the perceptron's weights are updated during training. The gradual stabilization of the weights shows that the learning algorithm has converged and effectively separates the two classes

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(
    range(1, perceptron.epochs + 1),
    perceptron.bias_history,
    marker='o',
    color='red'
)

plt.title("Bias Evolution")
plt.xlabel("Epoch")
plt.ylabel("Bias")
plt.grid(True)

plt.savefig(
    "bias_evolution.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

The bias changes throughout training to adjust the decision boundary for improved classification. As training progresses the bias converges to a stable value which means the model has reached an optimal decision threshold

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix

# Predict on test data
y_pred = perceptron.predict(X_test)

# Evaluation Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix")
print(cm)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")

plt.title("Confusion Matrix")
plt.savefig(
    "confusion_matrix.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

Shows that the majority of banknotes are classified correctly and has very few misclassifications. The high number of true positives and true negatives demonstrates the effectiveness of the perceptron

In [ ]:
learning_rates = [0.001, 0.01, 0.1]

models = {}
accuracies = {}

for lr in learning_rates:

    print("=" * 60)
    print(f"Training with Learning Rate = {lr}")

    model = Perceptron(learning_rate=lr, epochs=20)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    models[lr] = model
    accuracies[lr] = acc

    print(f"Accuracy = {acc:.4f}")

In [ ]:
plt.figure(figsize=(8,5))

for lr in learning_rates:
    plt.plot(
        range(1, len(models[lr].errors)+1),
        models[lr].errors,
        marker='o',
        label=f"LR = {lr}"
    )

plt.xlabel("Epoch")
plt.ylabel("Misclassified Samples")
plt.title("Learning Rate Comparison")
plt.legend()
plt.grid(True)

plt.savefig(
    "learning_rate_comparison.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

A very small learning rate results in slower learning, a larger learning rate converges more quickly but excessively large values may cause unstable updates or oscillations

In [ ]:
print("Accuracy Comparison\n")

for lr, acc in accuracies.items():
    print(f"Learning Rate = {lr:<6} Accuracy = {acc:.4f}")

In [ ]:
from sklearn.linear_model import Perceptron

X_vis = X.iloc[:, :2].values
y_vis = y.values

model = Perceptron(max_iter=1000, random_state=42)
model.fit(X_vis, y_vis)

x_min, x_max = X_vis[:,0].min()-1, X_vis[:,0].max()+1
y_min, y_max = X_vis[:,1].min()-1, X_vis[:,1].max()+1

xx, yy = np.meshgrid(
    np.arange(x_min, x_max, 0.02),
    np.arange(y_min, y_max, 0.02)
)

Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8,6))
plt.contourf(xx, yy, Z, alpha=0.3)

plt.scatter(
    X_vis[:,0],
    X_vis[:,1],
    c=y_vis,
    edgecolors='k'
)

plt.xlabel("Variance")
plt.ylabel("Skewness")
plt.title("Decision Boundary")
plt.savefig(
    "decision_boundary.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

The perceptron successfully learns a linear separator as most of the samples are classified on the correct side of the boundary

In [ ]:
import pandas as pd

training_summary = pd.DataFrame({
    "Parameter": [
        "Dataset Size",
        "Train/Test Split",
        "Learning Rate",
        "Epochs",
        "Final Weights",
        "Final Bias",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],
    "Value": [
        len(df),
        "80% / 20%",
        perceptron.learning_rate,
        perceptron.epochs,
        np.round(perceptron.weights, 4),
        round(perceptron.bias, 4),
        round(accuracy, 4),
        round(precision, 4),
        round(recall, 4),
        round(f1, 4)
    ]
})

training_summary

In [ ]:
epoch_table = pd.DataFrame({
    "Epoch": range(1, perceptron.epochs + 1),
    "Errors": perceptron.errors,
    "Variance Weight": [w[0] for w in perceptron.weight_history],
    "Skewness Weight": [w[1] for w in perceptron.weight_history],
    "Curtosis Weight": [w[2] for w in perceptron.weight_history],
    "Entropy Weight": [w[3] for w in perceptron.weight_history],
    "Bias": perceptron.bias_history
})

epoch_table


ADDITIONAL TASKS


1. Compare the Step and Sigmoid activation functions. Discuss why the Step Activation Function is not used in modern deep learning models.


| Feature         | Step Activation | Sigmoid Activation |
| --------------- | --------------- | ------------------ |
| Output          | 0 or 1          | 0 to 1             |
| Nature          | Binary          | Continuous         |
| Differentiable  | No              | Yes                |
| Backpropagation | Not Supported   | Supported          |
| Application     | Perceptron      | Neural Networks    |


The Step Activation Function is used in classical perceptrons for binary classification but is not differentiable, making it unsuitable for backpropagation. The Sigmoid Activation Function is differentiable, produces probability-like outputs, and is widely used in neural networks.

2. Compare the implementation with Scikit-learn’s Perceptron.


In [ ]:
from sklearn.linear_model import Perceptron as SKPerceptron
from sklearn.metrics import accuracy_score

sk_model = SKPerceptron(random_state=42)
sk_model.fit(X_train, y_train)

sk_pred = sk_model.predict(X_test)

print("Scikit-Learn Accuracy:", accuracy_score(y_test, sk_pred))

2. The Scikit-Learn Perceptron achieved accuracy similar to the custom implementation. It is optimized, easier to use, and requires less code.

3. Repeat the experiment using learning rates 0.001, 0.01 and 0.1.

| Learning Rate | Observation                       |
| ------------- | --------------------------------- |
| 0.001         | Slow learning                     |
| 0.01          | Stable convergence                |
| 0.1           | Faster learning but may fluctuate |

A learning rate of 0.01 provides the best balance between speed and stability.


4. Explain why a Single Layer Perceptron cannot solve the XOR problem.

| X₁ | X₂ | Output |
| -- | -- | ------ |
| 0  | 0  | 0      |
| 0  | 1  | 1      |
| 1  | 0  | 1      |
| 1  | 1  | 0      |

XOR is not linearly separable, so a single straight line cannot separate its classes. Therefore, a Single Layer Perceptron cannot solve the XOR problem.

5. Study the effect of feature normalization on model convergence.

Feature normalization scales all features to a similar range. This prevents features with larger values from dominating the learning process, resulting in faster and more stable convergence.

In [ ]:
!git init

In [ ]:
!git status